# 第12章 考えて、想像して、確かめる (Reasoner と Generator をつなぐ)

ここまでの章で、Cosmos 3 Edge の 2 つの部分を別々に使ってきました。

- **Reasoner**: 見て、考えて、言葉や座標で答える (第2〜6章)
- **Generator**: 行動や指示から、未来の映像を作る (第7〜11章)

1 つのモデルに両方が入っているのが Cosmos 3 の特徴です。最後の章では 2 つをつないで、
**「いくつかの行動の候補を想像の中で試し、その結果を見て一番よい行動を選ぶ」** という、世界モデルの一番大事な使い方を組み立てます。

1. 考える: Reasoner に、今の場面でとるべき行動を答えさせる
2. 想像する: Generator で、候補の行動ごとに未来の映像を作る
3. 確かめる: Reasoner に、それぞれの未来の映像を採点させて、行動を選ぶ
4. 応用: 生成した動画の自動チェック (何本か作って、物理的にもっともらしいものを選ぶ)

GPU メモリを節約するため、Generator と Reasoner は交互に読み込みます。全体で A100 なら 15 分ほど、L4 ではその数倍かかります。

## 準備

**ランタイム**: 「ランタイム → ランタイムのタイプを変更」で **L4** か **A100** の GPU を選んでください (T4 は対象外)。
A100 (40GB 以上) ではモデル全体を GPU に載せ、L4 (24GB) では使う部品だけを GPU に載せる設定 (CPU オフロード) に自動で切り替えます。

**安全フィルタ**: 生成には NVIDIA の安全フィルタ (`cosmos_guardrail`) を使います。使うには Hugging Face で
[nvidia/Cosmos-1.0-Guardrail](https://huggingface.co/nvidia/Cosmos-1.0-Guardrail) の利用申請をして承認を受け、
Colab の左の鍵のマーク (シークレット) に `HF_TOKEN` を登録しておいてください。承認を待たずに試すときは、下の `GUARDRAIL` の印を外します。

In [ ]:
# @title GPU を確かめる
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
if DEVICE == "cuda":
    GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(torch.cuda.get_device_name(0), f"{GPU_GB:.0f} GB")
    assert torch.cuda.get_device_capability()[0] >= 8, "Ampere 以降の GPU (L4 / A100 など) を選んでください。T4 では動きません。"
else:
    GPU_GB = 0
    print(f"CUDA の GPU が見つかりません ({DEVICE} で動かします。とても遅くなります)")

In [ ]:
# @title インストール (数分)
# Cosmos3OmniPipeline は Diffusers の main ブランチに入っている
!pip install -q -U "transformers>=5.18" accelerate av imageio imageio-ffmpeg huggingface_hub
!pip install -q "diffusers @ git+https://github.com/huggingface/diffusers.git" cosmos_guardrail

In [ ]:
# @title 安全フィルタの設定とサンプル素材を取ってくる関数
import os
import urllib.request
from pathlib import Path

GUARDRAIL = True  # @param {type:"boolean"}
try:  # Colab のシークレットに HF_TOKEN があれば使う
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass

ASSETS = Path(os.environ.get("COSMOS_ASSETS", "assets"))
ASSETS.mkdir(parents=True, exist_ok=True)
FW = "https://media.githubusercontent.com/media/NVIDIA/cosmos-framework/main/.agents/skills/cookbooks/cosmos3"
HF = "https://huggingface.co/nvidia/Cosmos3-Edge/resolve/main/assets"


def fetch(url):
    """URL の素材を assets/ に落として、そのパスを返す (2 回目からは落とさない)"""
    path = ASSETS / url.rsplit("/", 1)[-1]
    if not path.exists():
        urllib.request.urlretrieve(url, path)
    return str(path)

In [ ]:
# @title 生成モデル (Generator) を読み込む関数
import gc
import json

from diffusers import Cosmos3OmniPipeline
from diffusers.schedulers.scheduling_unipc_multistep import UniPCMultistepScheduler
from diffusers.utils import export_to_video
from IPython.display import Image as IPImage, Video, display
from PIL import Image, ImageOps


def load_image(path, max_side=10_000):
    """画像を RGB で読む (max_side より大きければ縮める)"""
    img = path if isinstance(path, Image.Image) else ImageOps.exif_transpose(Image.open(path))
    img = img.convert("RGB")
    img.thumbnail((max_side, max_side))
    return img


def load_pipe(flow_shift):
    """Cosmos3OmniPipeline を読み込む (2 回目からは読み込み済みのものを使う)。
    flow_shift は公式の推奨値: 画像→動画は 12.0、行動 (action) を使う生成は 10.0"""
    global pipe
    if "pipe" not in globals():
        pipe = Cosmos3OmniPipeline.from_pretrained(
            "nvidia/Cosmos3-Edge", torch_dtype=torch.bfloat16, enable_safety_checker=GUARDRAIL)
        if DEVICE == "cuda" and GPU_GB < 40:
            pipe.enable_model_cpu_offload()  # L4 など: 使う部品だけ GPU に載せる
        else:
            pipe.to(DEVICE)
    pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config, flow_shift=flow_shift, use_karras_sigmas=False)
    return pipe


def free_gpu():
    """読み込んだモデルを GPU から降ろす"""
    for name in ("pipe", "model", "processor"):
        globals().pop(name, None)
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()


def show_video(path, width=640):
    display(Video(path, embed=True, width=width))

In [ ]:
# @title 質問用の関数を用意する
import json
import re

import av
import numpy as np
from PIL import Image, ImageDraw, ImageOps
from transformers.video_utils import VideoMetadata

VIDEO_EXT = (".mp4", ".mov", ".webm", ".avi", ".mkv")


def load_image(path, max_side=1280):
    """画像を RGB で読む。大きすぎる画像はトークンが増えて遅くなるので縮める (座標は 0〜1000 の比率なので影響しない)"""
    img = path if isinstance(path, Image.Image) else ImageOps.exif_transpose(Image.open(path))
    img = img.convert("RGB")
    img.thumbnail((max_side, max_side))
    return img


def read_video(path, fps=4, max_side=640, max_frames=256):
    """動画を fps 枚/秒に間引いて読む。返すのはコマの配列と、各コマの時刻を伝えるための情報 (VideoMetadata)。
    Reasoner は各コマに「<1.5 seconds>」のような時刻を付けて読むので、時刻を答える問題 (第4章) にはこの情報が欠かせない"""
    frames, t_next = [], 0.0
    with av.open(path) as c:
        for f in c.decode(video=0):
            if f.time is None or f.time + 1e-6 < t_next:
                continue
            img = f.to_image()
            img.thumbnail((max_side, max_side))  # 1 コマのトークン数を抑える
            frames.append(np.asarray(img.convert("RGB")))
            t_next += 1 / fps
            if len(frames) >= max_frames:
                break
    n = len(frames)
    return np.stack(frames), VideoMetadata(total_num_frames=n, fps=fps, frames_indices=list(range(n)), duration=n / fps)


def ask(prompt, media=None, thinking=False, max_new_tokens=2048, fps=4, seed=0):
    """画像・動画 (なくてもよい) と質問を渡して、答えの文字列を返す。

    thinking=True のときは考えた過程 (<think>…</think>) を出してから答える。考えた過程は ask.last_think に残る。
    動画は fps 枚/秒で間引いて読む (推奨は 4)。
    """
    media = [] if media is None else media if isinstance(media, (list, tuple)) else [media]
    content, metadata = [], []
    for m in media:  # 画像・動画は質問文より前に置く (公式の推奨)
        if isinstance(m, str) and m.lower().endswith(VIDEO_EXT):
            frames, meta = read_video(m, fps=fps)
            content.append({"type": "video", "video": frames})
            metadata.append(meta)
        else:
            content.append({"type": "image", "image": load_image(m)})
    content.append({"type": "text", "text": prompt})
    # 動画は read_video で間引き済みなので、ここではもう間引かない
    video_kwargs = {"video_metadata": metadata, "do_sample_frames": False} if metadata else {}
    inputs = processor.apply_chat_template(
        [{"role": "user", "content": content}],
        tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt",
        enable_thinking=thinking, **video_kwargs,
    ).to(model.device)
    # 公式のプロンプトガイドのサンプリング設定 (考える / 考えない で値が違う)
    sampling = dict(temperature=0.6, top_p=0.95, top_k=20) if thinking else dict(temperature=0.7, top_p=0.8, top_k=20)
    torch.manual_seed(seed)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True, **sampling)
    text = processor.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    ask.last_think = ""
    if "</think>" in text:
        think, text = text.split("</think>", 1)
        ask.last_think = think.replace("<think>", "").strip()
    return text.strip()


def parse_json(text):
    """答えの中の JSON (```json … ``` で囲まれていてもよい) を取り出す。読めなければ None。
    小さなモデルの答えは、最後に余計な括弧が付く・真偽値の後ろに " が付く・最後に余計なカンマがある、
    といった崩れ方をするので、括弧の対応を数えて最初のまとまりだけを取り出し、よくある崩れを直してから読む"""
    m = re.search(r"```(?:json)?\s*(.*?)```", text, re.S)
    body = m.group(1) if m else text
    starts = [i for i in (body.find("["), body.find("{")) if i >= 0]
    if not starts:
        return None
    start = min(starts)
    depth, in_str, esc, end = 0, False, False, len(body)
    for k in range(start, len(body)):
        ch = body[k]
        if in_str:
            esc = (ch == "\\") and not esc
            if ch == '"' and not esc:
                in_str = False
            continue
        if ch == '"':
            in_str = True
        elif ch in "[{":
            depth += 1
        elif ch in "]}":
            depth -= 1
            if depth == 0:
                end = k + 1
                break
    chunk = body[start:end]
    fixed = re.sub(r'\b(true|false|null)"', r"\1", re.sub(r",\s*([\]}])", r"\1", chunk))
    for attempt in (chunk, fixed):
        try:
            return json.loads(attempt)
        except json.JSONDecodeError:
            pass
    print("(JSON として読めませんでした)")
    return None


def load_reasoner():
    """Reasoner を読み込む (生成の章で、Generator と入れ替えて使うとき用)"""
    global model, processor
    from transformers import AutoModelForImageTextToText, AutoProcessor
    model = AutoModelForImageTextToText.from_pretrained("nvidia/Cosmos3-Edge", dtype=torch.bfloat16, device_map="auto")
    processor = AutoProcessor.from_pretrained("nvidia/Cosmos3-Edge")

In [ ]:
# @title 行動 (action) を作る・描く関数
import numpy as np
import matplotlib.pyplot as plt
from diffusers import CosmosActionCondition

RAW = "https://raw.githubusercontent.com/NVIDIA/cosmos-framework/main/.agents/skills/cookbooks/cosmos3"
AV_PROMPT = "You are an autonomous vehicle planning system."
# AV の行動は「1 コマ前のカメラから見た、次のコマのカメラの位置と向き」を 9 個の数で表す。
#   [x, y, z, r1..r6]  x: 右, y: 下, z: 前 (OpenCV のカメラ座標)。r1..r6: 回転行列の 1 列目と 2 列目 (rot6d)
# 移動量はメートルを 1.35 倍した値 (公式の可視化コードと同じ換算)
AV_SCALE = 1.35


def rot_y(theta):
    """カメラの下向きの軸 (y) まわりの回転。theta が負だと左を向く"""
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]])


def from_rot6d(r):
    a = np.asarray(r[:3], float)
    b = np.asarray(r[3:6], float)
    c0 = a / np.linalg.norm(a)
    c1 = b - c0 * (c0 @ b)
    c1 /= np.linalg.norm(c1)
    return np.stack([c0, c1, np.cross(c0, c1)], axis=1)


def make_av_actions(speed_kmh=30.0, left_deg_per_s=0.0, accel=0.0, seconds=6.0, fps=10):
    """速さ (km/h)・左へ曲がる速さ (度/秒、右は負)・加速度 (m/s^2、減速は負) から AV の行動 (T, 9) を作る。
    left_deg_per_s には「時刻 t (秒) → 度/秒」の関数も渡せる (車線変更などに)"""
    rows, dt = [], 1 / fps
    for i in range(int(round(seconds * fps))):
        t = i * dt
        v = max(speed_kmh / 3.6 + accel * t, 0.0)
        d = v * dt
        yaw = left_deg_per_s(t) if callable(left_deg_per_s) else left_deg_per_s
        th = -np.radians(yaw) * dt if d > 0 else 0.0
        move = np.array([np.sin(th / 2) * d, 0.0, np.cos(th / 2) * d]) * AV_SCALE
        R = rot_y(th)
        rows.append(np.concatenate([move, R[:, 0], R[:, 1]]))
    return np.array(rows, np.float32)


def to_path(actions, scale=AV_SCALE):
    """行動 (T, 9) をつなげて、上から見た道筋 (T+1, 3) にする。x: 右, z: 前 (メートル)"""
    R, p, pts = np.eye(3), np.zeros(3), [np.zeros(3)]
    for a in np.asarray(actions, float):
        p = p + R @ (a[:3] / scale)
        R = R @ from_rot6d(a[3:9])
        pts.append(p.copy())
    return np.array(pts)


def plot_paths(paths, title="top-down view", fps=10):
    """{名前: 行動 (T, 9)} を上から見た図に描く。点は 1 秒ごと"""
    fig, ax = plt.subplots(figsize=(5, 5))
    for i, (name, actions) in enumerate(paths.items()):
        p = to_path(actions)
        ax.plot(p[:, 0], p[:, 2], lw=2, label=name)
        ax.scatter(p[::fps, 0], p[::fps, 2], s=12)
    ax.scatter([0], [0], c="k", marker="^", s=80, zorder=5)
    ax.set_xlabel("x: right (m)")
    ax.set_ylabel("z: forward (m)")
    ax.set_aspect("equal", adjustable="datalim")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
    ax.set_title(title)
    plt.show()


def tile_videos(videos, labels, out, fps, cols=2):
    """同じ長さの動画 (PIL 画像のリスト) を並べて 1 本にする"""
    from PIL import ImageDraw, ImageFont
    n = min(len(v) for v in videos)
    w, h = videos[0][0].size
    rows = (len(videos) + cols - 1) // cols
    font = ImageFont.load_default(size=max(16, w // 25))
    frames = []
    for t in range(n):
        canvas = Image.new("RGB", (w * cols, h * rows))
        for k, (v, label) in enumerate(zip(videos, labels)):
            tile = v[t].convert("RGB").resize((w, h))
            ImageDraw.Draw(tile).text((10, 8), label, fill="white", font=font, stroke_width=3, stroke_fill="black")
            canvas.paste(tile, ((k % cols) * w, (k // cols) * h))
        frames.append(canvas)
    export_to_video(frames, out, fps=fps, macro_block_size=1)
    return out

## 1. 考える: とるべき行動を答えさせる

車載カメラの 1 コマを見せて、4 つの候補 (直進・左折・右折・停止) のどれがよいかを、まず Reasoner に考えさせます。
これは「想像せずに、見ただけで決めた」答えです。

In [ ]:
start = load_image(fetch(f"{FW}/generator/action/assets/images/av_0.jpg"))
display(start.resize((416, 240)))

CANDIDATES = {
    "straight": make_av_actions(speed_kmh=40),
    "turn_left": make_av_actions(speed_kmh=20, left_deg_per_s=15),
    "turn_right": make_av_actions(speed_kmh=20, left_deg_per_s=-15),
    "stop": make_av_actions(speed_kmh=30, accel=-3.0),
}
plot_paths(CANDIDATES, "candidate actions")

free_gpu()
load_reasoner()
answer = ask(f"""You are an autonomous vehicle planning system. This is the current view from the vehicle's front camera.
Choose the best action for the next 6 seconds from: {list(CANDIDATES)}.
Return one JSON object: {{"action": "...", "reason": "one sentence"}}""", start, thinking=True)
first_choice = parse_json(answer) or {}
print("見ただけで選んだ行動:", first_choice)

## 2. 想像する: 候補ごとに未来の映像を作る

第8章と同じ方法で、4 つの候補それぞれの 6 秒先を作ります。

In [ ]:
free_gpu()
pipe = load_pipe(flow_shift=10.0)
futures = {}
for name, actions in CANDIDATES.items():
    result = pipe(
        prompt=AV_PROMPT,
        action=CosmosActionCondition(mode="forward_dynamics", chunk_size=len(actions), domain_name="av", resolution_tier=480,
                                     raw_actions=torch.as_tensor(actions), image=start, view_point="ego_view"),
        fps=10, num_inference_steps=30, guidance_scale=1.0, use_system_prompt=False,
        generator=torch.Generator(device=DEVICE).manual_seed(0),
    )
    futures[name] = f"ch12_future_{name}.mp4"
    export_to_video(result.video, futures[name], fps=10, macro_block_size=1)
    print("想像した:", name)

In [ ]:
# 4 本を並べて見る
import av as _av


def read_frames(path):
    with _av.open(path) as c:
        return [f.to_image() for f in c.decode(video=0)]


tile_videos([read_frames(p) for p in futures.values()], list(futures), "ch12_futures.mp4", fps=10)
show_video("ch12_futures.mp4", 832)

## 3. 確かめる: 未来の映像を採点させて、行動を選ぶ

Reasoner に、想像した 4 本の映像を 1 本ずつ見せて採点させます。
見るのは「ぶつからないか」「道から外れないか」「物理的にもっともらしいか」です (第6章の聞き方を組み合わせています)。

In [ ]:
free_gpu()
load_reasoner()
scores = {}
for name, path in futures.items():
    answer = ask("""This video is a predicted future from the front camera of the ego vehicle.
Judge the ego vehicle's behavior in this video. Return one JSON object with these keys:
"collision": true if the ego vehicle hits or nearly hits anything,
"on_road": true if the ego vehicle stays on a drivable road,
"plausible": true if the video is physically plausible,
"score": an integer from 1 (dangerous) to 5 (safe and reasonable),
"reason": one sentence.""", path, thinking=True)
    verdict = parse_json(answer)
    if isinstance(verdict, list):  # [{...}] と 1 つだけのリストで返すことがある
        verdict = verdict[0] if verdict else None
    scores[name] = verdict or {"score": 0, "reason": answer[:200]}
    print(f"{name:10s} {scores[name]}")

best = max(scores, key=lambda k: (scores[k].get("score") or 0) - 3 * bool(scores[k].get("collision")))
print("\n見ただけで選んだ行動:", first_choice.get("action"))
print("想像して確かめて選んだ行動:", best)
show_video(futures[best], 640)

「見ただけで選んだ行動」と「想像して確かめて選んだ行動」が違うことがあります。
想像を挟むと、「曲がった先に道がない」「止まれずに前の車に近づく」といった、**やってみないと分からない結果**を判断に入れられます。
実際のロボットや車では、選んだ行動の最初の少しだけを実行し、新しく見えた映像からまた 1 に戻ります (モデル予測制御の考え方)。

ただし、どちらも同じ 4B のモデルの推測です。想像した映像が間違っていれば、その採点も間違います。

## 4. 応用: 生成した動画を自動でチェックする

第7章の画像→動画では、seed によって出来がかなり変わりました。何本か作って、Reasoner に「物理的にもっともらしいか」を判定させれば、
人が全部見なくても良さそうなものを選べます。ここでは速さを優先して 256p (320×192) で 3 本作ります。

In [ ]:
free_gpu()
pipe = load_pipe(flow_shift=12.0)
json_prompt = json.loads(Path(fetch(f"{HF}/example_i2v_prompt.json")).read_text())
negative_prompt = json.loads(Path(fetch(f"{HF}/negative_prompt.json")).read_text())
json_prompt.update({"resolution": {"W": 320, "H": 192}, "aspect_ratio": "16,9", "duration": "2s", "fps": 24})
first_frame = load_image(fetch(f"{HF}/example_i2v_input.jpg"), max_side=10_000)

candidates = []
for seed in range(3):
    result = pipe(
        prompt=json.dumps(json_prompt), negative_prompt=json.dumps(negative_prompt), image=first_frame,
        num_frames=53, height=192, width=320, fps=24.0, num_inference_steps=20, guidance_scale=6.0,
        generator=torch.Generator(device=DEVICE).manual_seed(seed),
        add_resolution_template=False, add_duration_template=False,
    )
    path = f"ch12_i2v_seed{seed}.mp4"
    export_to_video(result.video, path, fps=24, macro_block_size=1)
    candidates.append(path)
tile_videos([read_frames(p) for p in candidates], [f"seed {s}" for s in range(3)], "ch12_i2v_candidates.mp4", fps=24, cols=3)
show_video("ch12_i2v_candidates.mp4", 960)

In [ ]:
free_gpu()
load_reasoner()
for path in candidates:
    verdict = ask("""Is this video physically plausible/possible according to your understanding of e.g. object permanence, shape constancy (objects maintain shape over time), continuous trajectories of objects? Assume it is the normal laws of physics.
(A) Possible
(B) Impossible
Answer with A or B, then one sentence explaining the most noticeable problem if any.""", path)
    print(f"{path}: {verdict}")

## この本のまとめ

| 章 | できること | 使う部分 |
| --- | --- | --- |
| 2 | 画像・動画を言葉にする | Reasoner |
| 3 | 物の位置を座標 (枠・点) で取る | Reasoner |
| 4 | 動画の出来事の時刻を取る | Reasoner |
| 5 | ロボットの作業を計画し、手先の道筋を出す | Reasoner |
| 6 | 運転シーンの危険予測と判断 | Reasoner |
| 7 | 画像から未来の動画を作る | Generator (+ Reasoner でプロンプト作り) |
| 8 | 車の行動から未来を予測する | Generator (forward dynamics) |
| 9 | 映像から車の動きを逆算する | Generator (inverse dynamics) |
| 10 | ロボットの手先の行動から未来を予測する | Generator (forward dynamics) |
| 11 | 見て、行動を決める | 方策モデル (Edge-Policy-DROID) |
| 12 | 想像して確かめてから行動を選ぶ | Reasoner + Generator |

次の一歩は、自分のロボット・自分の車のデータで追加学習すること (Cosmos Framework のポストトレーニング)、
そして Jetson などのエッジ機器で動かすこと (vLLM / TensorRT Edge-LLM) です。